In [ ]:
!pip install adapters -q

In [ ]:
import pandas as pd
from transformers import TrainingArguments, EarlyStoppingCallback, AutoTokenizer, set_seed
from datasets import Dataset
import joblib
import numpy as np
from google.colab import drive
import os
import json
import zipfile
from sklearn.metrics import f1_score, classification_report
import time
import torch
from transformers.trainer_utils import get_last_checkpoint
import adapters
from adapters import AutoAdapterModel, DoubleSeqBnConfig, AdapterTrainer
#DoubleSeqBnConfig = Houlsby

In [ ]:
drive.mount('/content/drive')

SPLIT_PATH = "/content/drive/MyDrive/thesis_results/SCOTBESS_splits/SCOTBESS_FULL_ANNOTATED_TERRA_LOW_SPLIT.csv"
MLB_PATH = "/content/drive/MyDrive/thesis_results/SCOTBESS_splits/scotbess_mlb.joblib"

output_dir = "/content/drive/MyDrive/thesis_results/SCOTBESS_EXPERIMENTS/SCOTBESS_DistilBERT_Houlsby"
os.makedirs(output_dir, exist_ok=True)


Mounted at /content/drive


Loading the dataset

In [ ]:
scotbess_df = pd.read_csv(SPLIT_PATH)

In [ ]:
scotbess_df_train = scotbess_df[scotbess_df["split"] == "train"].reset_index(drop=True)
scotbess_df_val = scotbess_df[scotbess_df["split"] == "validation"].reset_index(drop=True)
scotbess_df_test = scotbess_df[scotbess_df["split"] == "test"].reset_index(drop=True)

print("Train shape:", scotbess_df_train.shape)
print("Validation shape:", scotbess_df_val.shape)
print("Test shape:", scotbess_df_test.shape)

Train shape: (1340, 7)
Validation shape: (165, 7)
Test shape: (170, 7)


In [ ]:
mlb = joblib.load(MLB_PATH)

print("Number of labels:", len(mlb.classes_))
print(mlb.classes_)

Number of labels: 20
['Agricultural Land' 'Community and Economic Benefits'
 'Consultation, Transparency and Information' 'Cumulative Impact'
 'Decommissioning and Site Restoration' 'Emergency Planning and Response'
 'Fire and Explosion Risk' 'Grid Connection and Electrical Infrastructure'
 'Health and Wellbeing' 'Landscape, Visual and Heritage Impact'
 'Light Pollution' 'Noise' 'Planning Policy and Regulatory Compliance'
 'Project Need' 'Property Value'
 'Residential Proximity and Separation Distance' 'Site Selection'
 'Traffic' 'Water and Soil Contamination' 'Wildlife and Ecology']


In [ ]:
def parse_labels(value):
    if isinstance(value, list):
        return value
    if pd.isna(value):
        return []
    return json.loads(value)

for df in [scotbess_df_train, scotbess_df_val, scotbess_df_test]:
    df["label_list"] = df["labels"].apply(parse_labels)


In [ ]:
scotbess_y_train = mlb.transform(scotbess_df_train["label_list"])
scotbess_y_val = mlb.transform(scotbess_df_val["label_list"])
scotbess_y_test = mlb.transform(scotbess_df_test["label_list"])


In [ ]:
scotbess_y_train.shape, scotbess_y_val.shape, scotbess_y_test.shape


((1340, 20), (165, 20), (170, 20))

In [ ]:
scotbess_X_train = scotbess_df_train["final_masked_text"].fillna("").astype(str)
scotbess_X_val = scotbess_df_val["final_masked_text"].fillna("").astype(str)
scotbess_X_test = scotbess_df_test["final_masked_text"].fillna("").astype(str)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
# for DistilBERT max token length is 512 -  truncation will happen
def tokenize(texts):
    return tokenizer(texts.tolist(), padding="max_length", truncation=True, max_length=512)

train_enc = tokenize(scotbess_X_train)
dev_enc = tokenize(scotbess_X_val)
test_enc = tokenize(scotbess_X_test)

In [ ]:
y_train_bin = scotbess_y_train.astype(np.float32)
y_dev_bin = scotbess_y_val.astype(np.float32)
y_test_bin = scotbess_y_test.astype(np.float32)

print(mlb.classes_)
print(y_train_bin.shape)


['Agricultural Land' 'Community and Economic Benefits'
 'Consultation, Transparency and Information' 'Cumulative Impact'
 'Decommissioning and Site Restoration' 'Emergency Planning and Response'
 'Fire and Explosion Risk' 'Grid Connection and Electrical Infrastructure'
 'Health and Wellbeing' 'Landscape, Visual and Heritage Impact'
 'Light Pollution' 'Noise' 'Planning Policy and Regulatory Compliance'
 'Project Need' 'Property Value'
 'Residential Proximity and Separation Distance' 'Site Selection'
 'Traffic' 'Water and Soil Contamination' 'Wildlife and Ecology']
(1340, 20)


In [ ]:
id2label = {i: label for i, label in enumerate(mlb.classes_)}
label2id = {label: i for i, label in enumerate(mlb.classes_)}

In [ ]:
train_dataset = Dataset.from_dict({
    "input_ids": train_enc["input_ids"],
    "attention_mask": train_enc["attention_mask"],
    "labels": y_train_bin.astype("float32")})

eval_dataset = Dataset.from_dict({
    "input_ids": dev_enc["input_ids"],
    "attention_mask": dev_enc["attention_mask"],
    "labels": y_dev_bin.astype("float32")})

test_dataset = Dataset.from_dict({
    "input_ids": test_enc["input_ids"],
    "attention_mask": test_enc["attention_mask"],
    "labels": y_test_bin.astype("float32")})

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    probs = 1 / (1 + np.exp(-logits))   # sigmoid
    preds = (probs >= 0.5).astype(int) #default for now

    labels = labels.astype(int)

    f1_micro = f1_score(labels, preds, average="micro", zero_division=0)
    f1_macro = f1_score(labels, preds, average="macro", zero_division=0)

    return {"f1_micro": f1_micro, "f1_macro": f1_macro}

In [ ]:
print("Train labels:", y_train_bin.shape)
print("Val labels:", y_dev_bin.shape)
print("Test labels:", y_test_bin.shape)
print("Number of labels:", len(mlb.classes_))

Train labels: (1340, 20)
Val labels: (165, 20)
Test labels: (170, 20)
Number of labels: 20


**Training function**

In [ ]:
def sync_cuda():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def reset_cuda_peak_memory():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()

#measure vram only for final best config
def get_peak_vram_gb():
    if not torch.cuda.is_available():
        return None

    torch.cuda.synchronize()
    return torch.cuda.max_memory_allocated() / (1024 ** 3)


def run_training(config, seed=0, evaluate_test=False, measure_vram=False, save_report=False):
    set_seed(seed)

    if measure_vram:
        reset_cuda_peak_memory()

    model = AutoAdapterModel.from_pretrained(config["base_model"])
    #remove unnecessary default/pretraining head as a check revealed it is present
    if "default" in model.heads:
        model.delete_head("default")

    #the adapters' library developers wrote on github that multilabel classification head is supported out of the box and can be implemented like this
    model.add_classification_head("scotbess", num_labels=len(mlb.classes_),  multilabel=True, id2label=id2label)

    adapter_config = DoubleSeqBnConfig(reduction_factor=config["reduction_factor"])
    model.add_adapter("scotbess", config=adapter_config, set_active=True)
    model.train_adapter("scotbess")

    #check whether the adapters are working
    print("Active adapters:", model.active_adapters)
    print(model.adapter_summary())
    #check the classification head
    print("Trainable classification/head parameters:")
    for name, param in model.named_parameters():
        if param.requires_grad and ("head" in name.lower() or "classifier" in name.lower() or "classification" in name.lower()):
           print(name, param.numel())

    #for the final statistics
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())

    training_args = TrainingArguments(
        output_dir=config["output_dir"],

        learning_rate=config["learning_rate"],
        per_device_train_batch_size=config["batch_size"],
        per_device_eval_batch_size=config["batch_size"],

        num_train_epochs=config["num_train_epochs"],
        weight_decay=config["weight_decay"],
        warmup_ratio=config["warmup_ratio"],

        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="epoch",

        load_best_model_at_end=True,
        metric_for_best_model="eval_f1_macro",
        greater_is_better=True,

        save_total_limit=2,
        fp16=True,
        report_to="none")
    #adapter trainer, as recommended in the library's documentation
    trainer = AdapterTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=config["early_stopping_patience"])])

    #check
    print("Active adapters after trainer creation:", trainer.model.active_adapters)

    #for resuming if something goes wrong//collab's runtime gets disconnected
    last_checkpoint = None
    if os.path.isdir(config["output_dir"]):
        last_checkpoint = get_last_checkpoint(config["output_dir"])

    if last_checkpoint is not None:
        print(f"Resuming from checkpoint: {last_checkpoint}")
    else:
        print("Starting training from scratch.")


    sync_cuda()
    train_start = time.perf_counter()
    #includes training + epoch validation + checkpoint saving + early stopping + loading best model
    trainer.train(resume_from_checkpoint=last_checkpoint)

    sync_cuda()
    train_time_sec = time.perf_counter() - train_start
    if measure_vram:
        training_peak_vram_gb = get_peak_vram_gb()
    else:
        training_peak_vram_gb = None

    sync_cuda()
    val_start = time.perf_counter()

    val_results = trainer.evaluate(eval_dataset, metric_key_prefix="val")

    sync_cuda()
    val_eval_time_sec = time.perf_counter() - val_start

    result = {
        "model": config["base_model"],
        "dataset": "Scot-BESS",
        "method": "houlsby",
        "seed": seed,

        "learning_rate": config["learning_rate"],
        "batch_size": config["batch_size"],
        "num_train_epochs": config["num_train_epochs"],

        "best_checkpoint": trainer.state.best_model_checkpoint,
        "best_metric": trainer.state.best_metric,
        "actual_epochs_trained": trainer.state.epoch,


        "train_time_sec": train_time_sec,
        "val_eval_time_sec": val_eval_time_sec,

        "training_peak_vram_gb": training_peak_vram_gb,

        "trainable_params": trainable_params,
        "total_params": total_params,

        "val_f1_macro": val_results["val_f1_macro"],
        "val_f1_micro": val_results["val_f1_micro"]}

    if evaluate_test:
        sync_cuda()
        test_start = time.perf_counter()

        #single forward pass, gives metrics + raw predictions
        test_pred_output = trainer.predict(test_dataset)

        sync_cuda()
        test_eval_time_sec = time.perf_counter() - test_start

        #derive predictions, needed for classification report
        test_probs = 1 / (1 + np.exp(-test_pred_output.predictions))
        test_binary_preds = (test_probs >= 0.5).astype(int)
        gold_labels = (test_pred_output.label_ids >= 0.5).astype(int)

        test_metrics = test_pred_output.metrics

        result.update({
            "test_eval_time_sec": test_eval_time_sec,
            "test_inference_per_sample_ms": (test_eval_time_sec / len(test_dataset)) * 1000,
            "test_f1_macro": test_metrics["test_f1_macro"],
            "test_f1_micro": test_metrics["test_f1_micro"],
            "avg_predicted_labels": float(test_binary_preds.sum(axis=1).mean()),
            "avg_gold_labels": float(gold_labels.sum(axis=1).mean()),})

        if save_report:
            report_dict = classification_report(
                gold_labels, test_binary_preds,
                target_names=mlb.classes_, zero_division=0, output_dict=True)
            report_df = pd.DataFrame(report_dict).T
            report_path = os.path.join(output_dir, f"classification_report_seed_{seed}.csv")
            report_df.to_csv(report_path)
            print(f"Classification report saved to {report_path}")

            #saving raw arrays for possible future analysis
            predictions_path = os.path.join(output_dir, f"test_predictions_seed_{seed}.npz")
            np.savez_compressed(
                predictions_path,
                y_true=gold_labels,
                y_pred=test_binary_preds,
                y_prob=test_probs,
                label_names=np.array(mlb.classes_),
                threshold=np.array([0.5]))
            result["test_predictions_path"] = predictions_path
            print(f"Predictions saved to {predictions_path}")

            #for saving the adapter
            adapter_save_path = os.path.join(config["output_dir"], "final_Houlsby_adapter_with_head")
            trainer.model.save_adapter(adapter_save_path, "scotbess", with_head=True)
            result["saved_adapter_path"] = adapter_save_path
            print(f"Adapter + head saved to {adapter_save_path}")


    result["total_measured_time_sec"] = (result["train_time_sec"] + result["val_eval_time_sec"] + result.get("test_eval_time_sec", 0))

    return result

**Hyperparameter search**

In [ ]:
#fixed params
base_config = {
    "output_dir": os.path.join(output_dir, "search"),
    "base_model": "distilbert-base-uncased",
    "tokenizer_name": "distilbert-base-uncased",

    "max_length": 512,
    "num_train_epochs": 10,
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
    "early_stopping_patience": 3,

    "reduction_factor": 8}  #the default one is 16, but since DistilBERT is already a smaller model I will go with 8, this is also the choice of  Razuvayevskaya et al. (2024)

#small search on the most relevant hyperparameters
learning_rates = [1e-4, 2e-4, 5e-4] #1e-4 is recommmended in the adapters library documentation, 2e-4 is used by Razuvayevskaya et al. (2024)
batch_sizes = [8, 16]


search_results_path = os.path.join(output_dir, "search_results.csv")
search_results = []

for lr in learning_rates:
    for bs in batch_sizes:
        config = base_config.copy()
        config["learning_rate"] = lr
        config["batch_size"] = bs
        config["output_dir"] = (f"{base_config['output_dir']}/lr_{lr}_bs_{bs}")

        #skipping already-completed configs on resume
        if os.path.exists(search_results_path):
            existing = pd.read_csv(search_results_path)
            already_done = existing[
                (existing["learning_rate"] == lr) &
                (existing["batch_size"] == bs)]
            if len(already_done) > 0:
                print(f"Skipping lr={lr}, bs={bs} (already done)")
                search_results.append(already_done.iloc[0].to_dict())
                continue

        print("=" * 80)
        print(f"Running DistilBERT: lr={lr}, batch_size={bs}")
        print("=" * 80)

        result = run_training(config, seed=0)
        search_results.append(result)

        #saving incrementally after every config
        pd.DataFrame(search_results).to_csv(search_results_path, index=False)

search_results_df = pd.DataFrame(search_results)
search_results_df = search_results_df.sort_values("val_f1_macro", ascending=False).reset_index(drop=True)
search_results_df.to_csv(search_results_path, index=False)
#!!! train_time_sec in search results is unreliable due to checkpoint resumption !!!
# !!!timing is only reported from the final seed runs - I ensured the run is not resumed

#the warning about adapters can be ignored, based on the check the adapters work
#the warining about early stopping can also be ignored, it works

search_results_df

Running DistilBERT: lr=0.0001, batch_size=8


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Active adapters: Stack[scotbess]
Name                     Architecture         #Param      %Param  Active   Train
--------------------------------------------------------------------------------
scotbess                 bottleneck        1,779,840       2.682       1       1
--------------------------------------------------------------------------------
Full model                                66,362,880     100.000               0
Trainable classification/head parameters:
heads.scotbess.1.weight 589824
heads.scotbess.1.bias 768
heads.scotbess.4.weight 15360
heads.scotbess.4.bias 20
Active adapters after trainer creation: Stack[scotbess]
Starting training from scratch.


Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro
1,0.579100,0.474045,0.568935,0.410207
2,0.467900,0.449547,0.554591,0.397034
3,0.412600,0.394022,0.641916,0.492478
4,0.365800,0.360852,0.716467,0.574095
5,0.339600,0.351713,0.710364,0.575482
6,0.319700,0.335475,0.732866,0.613549
7,0.304000,0.327904,0.752665,0.633413
8,0.292800,0.326969,0.748421,0.631720
9,0.288400,0.318746,0.763199,0.659432
10,0.282600,0.318375,0.759958,0.653589


early stopping required metric_for_best_model, but did not find eval_f1_macro so early stopping is disabled


...

Running DistilBERT: lr=0.0005, batch_size=16


Active adapters: Stack[scotbess]
Name                     Architecture         #Param      %Param  Active   Train
--------------------------------------------------------------------------------
scotbess                 bottleneck        1,779,840       2.682       1       1
--------------------------------------------------------------------------------
Full model                                66,362,880     100.000               0
Trainable classification/head parameters:
heads.scotbess.1.weight 589824
heads.scotbess.1.bias 768
heads.scotbess.4.weight 15360
heads.scotbess.4.bias 20
Active adapters after trainer creation: Stack[scotbess]
Starting training from scratch.


Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro
1,0.545800,0.466090,0.573589,0.389117
2,0.418400,0.365151,0.709500,0.564090
3,0.339600,0.338133,0.733297,0.613582
4,0.290400,0.313508,0.772794,0.686237
5,0.257600,0.294982,0.769231,0.704275
6,0.229800,0.289949,0.791052,0.722670
7,0.204300,0.292460,0.794162,0.718246
8,0.184400,0.289967,0.786469,0.694116
9,0.169700,0.279494,0.795939,0.730535
10,0.158100,0.281233,0.793033,0.721192


early stopping required metric_for_best_model, but did not find eval_f1_macro so early stopping is disabled


,model,dataset,method,seed,learning_rate,batch_size,num_train_epochs,best_checkpoint,best_metric,actual_epochs_trained,train_time_sec,val_eval_time_sec,training_peak_vram_gb,trainable_params,total_params,val_f1_macro,val_f1_micro,total_measured_time_sec
0,distilbert-base-uncased,Scot-BESS,houlsby,0,0.0005,8,10,/content/drive/MyDrive/thesis_results/SCOTBESS...,0.753165,10.0,158.510971,0.797481,None,2385812,68748692,0.753165,0.811253,159.308452
1,distilbert-base-uncased,Scot-BESS,houlsby,0,0.0005,16,10,/content/drive/MyDrive/thesis_results/SCOTBESS...,0.730535,10.0,153.579605,0.777696,None,2385812,68748692,0.730535,0.795939,154.357301
2,distilbert-base-uncased,Scot-BESS,houlsby,0,0.0002,8,10,/content/drive/MyDrive/thesis_results/SCOTBESS...,0.715413,10.0,158.453092,0.812349,None,2385812,68748692,0.715413,0.792839,159.265441
3,distilbert-base-uncased,Scot-BESS,houlsby,0,0.0002,16,10,/content/drive/MyDrive/thesis_results/SCOTBESS...,0.688329,10.0,153.015219,0.794256,None,2385812,68748692,0.688329,0.776242,153.809475
4,distilbert-base-uncased,Scot-BESS,houlsby,0,0.0001,8,10,/content/drive/MyDrive/thesis_results/SCOTBESS...,0.659432,10.0,192.827836,0.809180,None,2385812,68748692,0.659432,0.763199,193.637015
5,distilbert-base-uncased,Scot-BESS,houlsby,0,0.0001,16,10,/content/drive/MyDrive/thesis_results/SCOTBESS...,0.596031,10.0,156.084243,0.861013,None,2385812,68748692,0.596031,0.723769,156.945256


**Best configuration**

In [ ]:
best_row = search_results_df.iloc[0]

best_lr = float(best_row["learning_rate"])
best_batch_size = int(best_row["batch_size"])

print("Best learning rate:", best_lr)
print("Best batch size:", best_batch_size)
print("Best validation macro-F1:", best_row["val_f1_macro"])
print("Best checkpoint:", best_row["best_checkpoint"])

Best learning rate: 0.0005
Best batch size: 8
Best validation macro-F1: 0.7531654284054219
Best checkpoint: /content/drive/MyDrive/thesis_results/SCOTBESS_EXPERIMENTS/SCOTBESS_DistilBERT_Houlsby/search/lr_0.0005_bs_8/checkpoint-1512


In [ ]:
best_config = base_config.copy()
best_config["learning_rate"] = best_lr
best_config["batch_size"] = best_batch_size
best_config["selection_metric"] = "val_f1_macro"
best_config["best_validation_macro_f1"] = float(best_row["val_f1_macro"])
best_config["best_validation_micro_f1"] = float(best_row["val_f1_micro"])
best_config["best_checkpoint_from_search"] = best_row["best_checkpoint"]

best_config_path = os.path.join(output_dir, "best_config.json")

with open(best_config_path, "w") as f:
    json.dump(best_config, f, indent=2)


**Final run (test set) on the best found configuration**

In [ ]:
with open(best_config_path, "r") as f:
    final_config = json.load(f)

In [ ]:
test_output_dir = os.path.join(output_dir, "test")
os.makedirs(test_output_dir, exist_ok=True)

test_results_path = os.path.join(test_output_dir, "SCOTBESS_DistilBERT_Houlsby_test_results.csv")
test_results = []

for seed in [0, 1, 2]:
    config = final_config.copy()
    config["seed"] = seed
    config["output_dir"] = (os.path.join(test_output_dir, f"SCOTBESS_DistilBERT_Houlsby_test_seed_{seed}"))

    # skips already-completed seeds on resume
    if os.path.exists(test_results_path):
        existing = pd.read_csv(test_results_path)
        already_done = existing[existing["seed"] == seed]
        if len(already_done) > 0:
            print(f"Skipping seed={seed} (already done)")
            test_results.append(already_done.iloc[0].to_dict())
            continue

    print("=" * 80)
    print(f"Final run: seed={seed}, lr={final_config['learning_rate']}, batch_size={final_config['batch_size']}")
    print("=" * 80)
    result = run_training(config, seed=seed, evaluate_test=True, measure_vram=True, save_report = True)
    test_results.append(result)

    #saves incrementally after every seed
    pd.DataFrame(test_results).to_csv(test_results_path, index=False)

test_results_df = pd.DataFrame(test_results)
test_results_df.to_csv(test_results_path, index=False)
test_results_df

Final run: seed=0, lr=0.0005, batch_size=8


Active adapters: Stack[scotbess]
Name                     Architecture         #Param      %Param  Active   Train
--------------------------------------------------------------------------------
scotbess                 bottleneck        1,779,840       2.682       1       1
--------------------------------------------------------------------------------
Full model                                66,362,880     100.000               0
Trainable classification/head parameters:
heads.scotbess.1.weight 589824
heads.scotbess.1.bias 768
heads.scotbess.4.weight 15360
heads.scotbess.4.bias 20
Active adapters after trainer creation: Stack[scotbess]
Starting training from scratch.


Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro
1,0.529300,0.451765,0.547634,0.363500
2,0.389100,0.347716,0.717226,0.589941
3,0.315000,0.321433,0.744462,0.650857
4,0.266600,0.303229,0.767296,0.682625
5,0.229500,0.285897,0.783158,0.701922
6,0.190800,0.283052,0.797237,0.746141
7,0.161400,0.281919,0.804402,0.744586
8,0.137400,0.284889,0.802237,0.732801
9,0.119700,0.276950,0.810645,0.750459
10,0.106400,0.279258,0.808269,0.742136


early stopping required metric_for_best_model, but did not find eval_f1_macro so early stopping is disabled


Classification report saved to /content/drive/MyDrive/thesis_results/SCOTBESS_EXPERIMENTS/SCOTBESS_DistilBERT_Houlsby/classification_report_seed_0.csv
Predictions saved to /content/drive/MyDrive/thesis_results/SCOTBESS_EXPERIMENTS/SCOTBESS_DistilBERT_Houlsby/test_predictions_seed_0.npz
Adapter + head saved to /content/drive/MyDrive/thesis_results/SCOTBESS_EXPERIMENTS/SCOTBESS_DistilBERT_Houlsby/test/SCOTBESS_DistilBERT_Houlsby_test_seed_0/final_Houlsby_adapter_with_head
Final run: seed=1, lr=0.0005, batch_size=8


...

Active adapters: Stack[scotbess]
Name                     Architecture         #Param      %Param  Active   Train
--------------------------------------------------------------------------------
scotbess                 bottleneck        1,779,840       2.682       1       1
--------------------------------------------------------------------------------
Full model                                66,362,880     100.000               0
Trainable classification/head parameters:
heads.scotbess.1.weight 589824
heads.scotbess.1.bias 768
heads.scotbess.4.weight 15360
heads.scotbess.4.bias 20
Active adapters after trainer creation: Stack[scotbess]
Starting training from scratch.


Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro
1,0.530600,0.440980,0.573358,0.372261
2,0.386000,0.354173,0.711528,0.569150
3,0.313100,0.322566,0.744212,0.641743
4,0.266200,0.297645,0.781963,0.707827
5,0.229600,0.281541,0.795478,0.734297
6,0.192200,0.275889,0.804082,0.748967
7,0.161700,0.275942,0.803030,0.748023
8,0.139500,0.276676,0.802638,0.742894
9,0.120300,0.276134,0.802450,0.749381
10,0.106200,0.275996,0.794872,0.736214


early stopping required metric_for_best_model, but did not find eval_f1_macro so early stopping is disabled


Classification report saved to /content/drive/MyDrive/thesis_results/SCOTBESS_EXPERIMENTS/SCOTBESS_DistilBERT_Houlsby/classification_report_seed_2.csv
Predictions saved to /content/drive/MyDrive/thesis_results/SCOTBESS_EXPERIMENTS/SCOTBESS_DistilBERT_Houlsby/test_predictions_seed_2.npz
Adapter + head saved to /content/drive/MyDrive/thesis_results/SCOTBESS_EXPERIMENTS/SCOTBESS_DistilBERT_Houlsby/test/SCOTBESS_DistilBERT_Houlsby_test_seed_2/final_Houlsby_adapter_with_head


,model,dataset,method,seed,learning_rate,batch_size,num_train_epochs,best_checkpoint,best_metric,actual_epochs_trained,...,val_f1_micro,test_eval_time_sec,test_inference_per_sample_ms,test_f1_macro,test_f1_micro,avg_predicted_labels,avg_gold_labels,test_predictions_path,saved_adapter_path,total_measured_time_sec
0,distilbert-base-uncased,Scot-BESS,houlsby,0,0.0005,8,10,/content/drive/MyDrive/thesis_results/SCOTBESS...,0.750459,10.0,...,0.810645,0.874710,5.145352,0.734034,0.795217,5.888235,5.917647,/content/drive/MyDrive/thesis_results/SCOTBESS...,/content/drive/MyDrive/thesis_results/SCOTBESS...,159.888004
1,distilbert-base-uncased,Scot-BESS,houlsby,1,0.0005,8,10,/content/drive/MyDrive/thesis_results/SCOTBESS...,0.745397,10.0,...,0.801184,0.975046,5.735564,0.728321,0.788637,6.300000,5.917647,/content/drive/MyDrive/thesis_results/SCOTBESS...,/content/drive/MyDrive/thesis_results/SCOTBESS...,161.105822
2,distilbert-base-uncased,Scot-BESS,houlsby,2,0.0005,8,10,/content/drive/MyDrive/thesis_results/SCOTBESS...,0.749381,10.0,...,0.802450,0.871341,5.125537,0.740955,0.802605,5.823529,5.917647,/content/drive/MyDrive/thesis_results/SCOTBESS...,/content/drive/MyDrive/thesis_results/SCOTBESS...,161.910716


In [ ]:
test_summary_df = test_results_df[[
    "test_f1_macro",
    "test_f1_micro",
    "avg_predicted_labels",
    "avg_gold_labels",
    "train_time_sec",
    "val_eval_time_sec",
    "test_eval_time_sec",
    "test_inference_per_sample_ms",
    "training_peak_vram_gb",
    "actual_epochs_trained",
    "trainable_params",
    "total_params",
    "total_measured_time_sec"]].agg(["mean", "std"])

test_summary_path =  os.path.join(test_output_dir, "SCOTBESS_DistilBERT_Houlsby_test_results_summary.csv")
test_summary_df.to_csv(test_summary_path)

test_summary_df

,test_f1_macro,test_f1_micro,avg_predicted_labels,avg_gold_labels,train_time_sec,val_eval_time_sec,test_eval_time_sec,test_inference_per_sample_ms,training_peak_vram_gb,actual_epochs_trained,trainable_params,total_params,total_measured_time_sec
mean,0.734437,0.795486,6.003922,5.917647,159.235707,0.825442,0.907032,5.335484,0.977190,10.0,2385812.0,68748692.0,160.968181
std,0.006327,0.006988,0.258444,0.000000,1.016439,0.028383,0.058926,0.346621,0.001555,0.0,0.0,0.0,1.018357
